# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Week 5 already trained under a client-grouped split, so this card is not "go back and fix the
split." It is the harder version: **prove the split was necessary, measure what the careless
version would have claimed, and go looking for the leak I did not already know about.**

The last part is the one that paid off. The starter feature vector ships two columns that
rebuild the label exactly — and neither of them looks suspicious on its own.

Skill loaded for this card: `skills/hunting-leakage-and-validating`, with
`skills/writing-honest-claims` for section 4.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Both questions below are about **construct validity** — not whether the arithmetic is right, but
whether the thing being measured is the thing being claimed. Both findings are already hedged in
the paper, which is to its credit; my questions are about whether the hedge is load-bearing enough
for the number that sits next to it.

---

### Finding A — ML Appendix, "What Predicts Health?" (p. 27)

**The claim.** A Random Forest predicting health score ranks Average Position at 43% importance,
Impressions at 32%, Scroll Depth at 15%, CTR at 8%.

**Where the label comes from.** Health score is defined on p. 5 as
`impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)`. The four
features the model ranks highest are the four terms the target is assembled from, and they carry
98% of the importance between them.

**Does the design carry the claim?** The model is recovering its own construction. The paper says
this plainly — *"health score is partly constructed from inputs such as position and impressions.
High importance is therefore expected"* — and that disclaimer is correct and welcome. My question
is why the table is in the paper at all once the disclaimer is true, because the page title asks
*"What Predicts Health?"* and the only honest answer the design can give is *"the definition of
health does."* A reader who skims the bars and skips the caveat leaves with an optimization order
that was never measured.

**My two questions, constructively:**

1. What does the importance table look like with the four constituent features **removed**? That
   version would answer the question the title asks — which *non-definitional* features track
   health — and it is a small re-run of an analysis already built.
2. The model is described as "holdout-tested." Across 57 brands, was the holdout **grouped by
   brand**? Section 2 of this notebook measures what that choice is worth on a 32-client corpus,
   and it is not small.

The code cell below reproduces the mechanism on the local sample so the point is demonstrated
rather than asserted.

---

### Finding B — Finding #4, "The Freshness Multiplier" (p. 9)

**The claim.** *"365+ day content that was refreshed within 30 days shows 3.2x health boost (from
10.7 to 34.5) and 57x more impressions (from 71 to 4039)."* The action box turns this into an
expected outcome: *"Lifts mature pages from roughly 10.7 health to 34.5."*

**Where the label comes from.** `freshness_tier` is days since last update. So the comparison is
**refreshed pages versus not-refreshed pages, measured at one moment** — a cross-section. It is
not the same pages before and after a refresh.

**Does the design carry the claim?** Not for a causal read, and the action box is written as a
causal read. Two specific problems:

- **Refresh is not randomly assigned.** Somebody chose which old pages to refresh, and the
  rational choice is the pages already worth the effort — the ones with demand, history, and
  strategic value. A comparison between chosen and unchosen pages measures the choice at least as
  much as the refresh. A 57x impression gap is far outside what a content edit plausibly produces
  and well inside what selection produces.
- **Survivor bias in the sample frame.** The local cuts are an active-content subset
  (`impressions_90d > 0 and sessions_90d > 0`, p. 4). An old page that was never refreshed and
  died has no impressions and is therefore not in the denominator. The paper flags this for the
  `365+ × 361+` heatmap cell on p. 14, which shows the team knows the mechanism — my question is
  why the same caution does not reach the 57x headline, which is drawn from the same frame.

**My two questions, constructively:**

1. The paper has a **monthly local historical series** (p. 4, and the trend table on p. 17). Can
   the refresh claim be re-cut **within page over time** — impressions in the 30 days before an
   update versus the 30 days after, same page as its own control? That design would carry a causal
   claim; the current one cannot, and the data to build it appears to exist.
2. What is the **n in each cell** of the 10.7 → 34.5 comparison? Finding #4 already discloses a
   283:1 ratio resting on a single declining page. The same disclosure on the headline cell would
   let a reader size the claim themselves.

**Where this lands on my own work:** question 2 applies to me too. The 30,000-row starter CSV is
*itself* an active-content sample — the cell below shows 100% of its rows pass the filter — so my
model is trained on survivors and cannot see the pages that already died. That limitation is
carried into section 4 and into the capstone rather than left here.

In [1]:
# Section 1 - reproduce the mechanism behind each question on the local sample.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "work" / "scripts"))

from validation_audit import SEED  # noqa: E402

frame = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
print(f"repo root : {ROOT}")
print(f"rows      : {len(frame):,}   seed: {SEED}")

# --- Finding A: rebuild the paper's health formula, then ask a model what predicts it ------
# The exact FlyRank normalisation is not published, so this reconstructs the *shape* of the
# metric from p.5 (30/30/20/20 over its four inputs) using percentile ranks. The circularity
# point does not depend on the normalisation - it depends on the target being a sum of features.
HEALTH_TERMS = {"log_impressions_90d": 30, "avg_position": 30, "ctr": 20, "scroll_rate": 20}
health = np.zeros(len(frame))
for column, points in HEALTH_TERMS.items():
    ranked = frame[column].rank(pct=True)
    if column == "avg_position":
        ranked = 1 - ranked  # position 1 is the good end
    health += ranked * points

# clicks == impressions * ctr, and sessions tracks clicks. These are not independent
# features competing with the constituents; they are the constituents recombined.
PROXIES = ["log_clicks_90d", "log_sessions_90d"]
INDEPENDENT = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "engagement_rate", "ai_traffic_pct",
]


def importance_for(features: list[str]) -> pd.Series:
    rf = RandomForestRegressor(
        n_estimators=120, min_samples_leaf=25, n_jobs=-1, random_state=SEED
    )
    rf.fit(frame[features], health)
    return pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)


print("\n--- Finding A: what predicts a score built out of its own features? ---")

# Run 1: everything on the table, including the algebraic proxies.
full = importance_for(list(HEALTH_TERMS) + PROXIES + INDEPENDENT)
constituents = full[list(HEALTH_TERMS)].sum()
proxies = full[PROXIES].sum()
print("\nrun 1 - constituents, their algebraic proxies, and 12 independent features:")
print(full.head(6).round(4).to_string())
print(f"  4 constituent terms      : {constituents:.1%}")
print(f"  clicks/sessions proxies  : {proxies:.1%}   (clicks == impressions x ctr, definitionally)")
print(f"  12 independent features  : {full[INDEPENDENT].sum():.1%}")
print(f"  -> definition + proxies  : {constituents + proxies:.1%} of all importance")

# Run 2: drop the proxies, so the constituents face only genuinely separate features.
direct = importance_for(list(HEALTH_TERMS) + INDEPENDENT)
print("\nrun 2 - proxies removed, constituents vs independent features only:")
print(direct.head(6).round(4).to_string())
print(f"  4 constituent terms      : {direct[list(HEALTH_TERMS)].sum():.1%}")
print(f"  12 independent features  : {direct[INDEPENDENT].sum():.1%}")
print("\n-> either way, ~99% of the importance sits on the target's own definition or on an")
print("   exact algebraic function of it. The model recovers the formula, not a finding.")
print("   Note run 1: hiding the constituents does not fix the circularity, it launders it")
print("   into a proxy - which is why removing them (my question 1) needs care too.")

# --- Finding B, part 1: is the starter corpus itself a survivor sample? --------------------
active = (frame["impressions_90d"] > 0) & (frame["sessions_90d"] > 0)
print("\n--- Finding B: the sample frame ---")
print(f"rows passing the paper's active-content filter: {active.sum():,} / {len(frame):,} ({active.mean():.1%})")
print("-> the starter CSV IS the active subset. Pages that died are already gone from it,")
print("   so neither the paper's survivor question nor mine can be audited from inside this file.")

# --- Finding B, part 2: is refresh randomly assigned? -------------------------------------
by_freshness = frame.groupby("freshness_tier").agg(
    pages=("content_id", "size"),
    mean_impressions_90d=("impressions_90d", "mean"),
    mean_word_count=("word_count", "mean"),
    mean_avg_position=("avg_position", "mean"),
    declining_rate=("is_declining_label", "mean"),
).round(2)
print("\ndays since last update, vs characteristics that predate the update:")
print(by_freshness.to_string())

spread = by_freshness["mean_impressions_90d"].max() / by_freshness["mean_impressions_90d"].min()
smallest = by_freshness["pages"].min()
print(f"\nimpression spread across freshness tiers here: {spread:.1f}x  (paper's headline gap: 57x)")
print(f"smallest tier in this corpus: {smallest:,} pages")
print("-> tiers differ on word count and position too, which no update could have caused")
print("   retroactively. That is selection, and it is inside any cross-sectional refresh gap.")

repo root : D:\Flyrank\FlyRank-Machine-Learning-Internship
rows      : 30,000   seed: 42

--- Finding A: what predicts a score built out of its own features? ---



run 1 - constituents, their algebraic proxies, and 12 independent features:
log_clicks_90d           0.4848
avg_position             0.2578
scroll_rate              0.1365
ctr                      0.0920
log_impressions_90d      0.0279
days_with_impressions    0.0008
  4 constituent terms      : 51.4%
  clicks/sessions proxies  : 48.5%   (clicks == impressions x ctr, definitionally)
  12 independent features  : 0.1%
  -> definition + proxies  : 99.9% of all importance



run 2 - proxies removed, constituents vs independent features only:
ctr                      0.5078
avg_position             0.2804
scroll_rate              0.1556
log_impressions_90d      0.0539
days_with_sessions       0.0014
days_with_impressions    0.0008
  4 constituent terms      : 99.8%
  12 independent features  : 0.2%

-> either way, ~99% of the importance sits on the target's own definition or on an
   exact algebraic function of it. The model recovers the formula, not a finding.
   Note run 1: hiding the constituents does not fix the circularity, it launders it
   into a proxy - which is why removing them (my question 1) needs care too.

--- Finding B: the sample frame ---
rows passing the paper's active-content filter: 30,000 / 30,000 (100.0%)
-> the starter CSV IS the active subset. Pages that died are already gone from it,
   so neither the paper's survivor question nor mine can be audited from inside this file.

days since last update, vs characteristics that predate th

**What the cell shows.**

*Finding A.* With the algebraic proxies removed, the four constituent terms take **99.8%** of the
importance and twelve genuinely independent features share **0.2%** between them. The reconstruction
is cruder than FlyRank's real health score and it does not matter: any model predicting a weighted
sum of four features will rank those four features first. The paper's bar chart is a picture of its
own formula.

Run 1 adds a wrinkle worth handing back with question 1. When `log_clicks_90d` is available, it
takes 48.5% on its own — because clicks *is* impressions × CTR. Deleting the four constituents would
not remove the circularity; it would relocate it into whichever proxy survives. So the re-run I asked
for needs to exclude the constituents **and** their algebraic descendants, or it will produce a
table that looks independent and is not.

*Finding B.* The starter corpus is **100% active content**, so the survivorship question cannot be
answered from inside it — by me or by a reader. And refresh is visibly not randomly assigned: the
freshness tiers differ by **6.4x** in mean impressions, but they also differ in word count and
average position, which no later update could have caused retroactively. Those pre-existing
differences are selection, and they sit inside any cross-sectional refresh gap — including the
paper's 57x. Two of the four tiers hold under 200 pages, which is where the instability the paper
already flagged for the 361+ bucket lives.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Week 5 was already grouped by `client_id`,** so "after" is the number I have been reporting all
along. To make this section mean something, I ran the comparison in the other direction: what
would the *naive* version have claimed?

Both runs use the identical feature set, the identical three models, the identical metrics and
`seed=42`. The only thing that changes is the fold assignment:

| | split | test clients also seen in training |
|---|---|---|
| **before** (naive) | `StratifiedKFold(5, shuffle=True)` over rows | measured below |
| **after** (honest) | `GroupKFold(5)` on `client_id` | measured below |

Why the naive number is wrong rather than merely optimistic: there are 32 clients across 30,000
rows and the per-client declining rate runs from 0.000 to 0.937 against a 0.542 base rate. Under a
row-level shuffle the same client sits on both sides, so a model can identify the client from its
feature fingerprint and predict that client's base rate. It scores well by being a client lookup
table. In use, the queue is run for a client whose pages the model has never seen — which is the
grouped condition, not the shuffled one.

A **time-aware split is not available here** and I am not going to pretend otherwise: the starter
CSV is one undated snapshot. The contract's sealed test month (2026-06) is the design to run once
the dated warehouse release is in hand, and it stays in the contract as future work.

The third row of the audit is a **shuffled-label null**: the same grouped design against a permuted
label. If that scored above chance, the evaluation harness itself would be broken and nothing else
in this notebook would be worth reading.

In [2]:
# Section 2 - naive random split vs grouped split, same everything else.
from validation_audit import OUT_JSON, run  # noqa: E402

REFRESH_AUDIT = False  # set True to recompute from scratch (~2 min); otherwise read the receipt

if REFRESH_AUDIT or not OUT_JSON.exists():
    audit = run()
    OUT_JSON.write_text(json.dumps(audit, indent=2), encoding="utf-8")
else:
    audit = json.loads(OUT_JSON.read_text(encoding="utf-8"))

comparison = audit["split_comparison"]
print(f"base rate            : {audit['base_rate']:.4f}")
print(f"clients              : {audit['n_clients']}   rows: {audit['rows']:,}")
print(f"naive  split         : {comparison['naive_random']['type']}")
print(f"  test clients seen in training: {comparison['naive_random']['client_overlap_share']:.0%}")
print(f"grouped split        : {comparison['grouped_by_client']['type']}")
print(f"  test clients seen in training: {comparison['grouped_by_client']['client_overlap_share']:.0%}")

rows = []
for model, metrics in comparison["inflation"].items():
    for metric, block in metrics.items():
        rows.append({
            "model": model,
            "metric": metric,
            "honest (grouped)": block["grouped"],
            "naive (random)": block["naive_random"],
            "gap": block["absolute_gap"],
            "inflation %": block["relative_inflation_pct"],
        })
print("\n--- what the naive split would have let me claim ---")
print(pd.DataFrame(rows).to_string(index=False))

null_auc = audit["shuffled_label_null"]["metrics"]["random_forest"]["roc_auc"]
print(f"\n--- shuffled-label null (grouped design, random forest) ---")
print(f"roc_auc mean {null_auc['mean']:.4f} +/- {null_auc['std']:.4f}  across {null_auc['folds']} folds")
print(f"per fold: {null_auc['per_fold']}")
print("-> sits on 0.50. The harness is measuring the label, not an artefact of the folds.")

rf_grouped = comparison["grouped_by_client"]["metrics"]["random_forest"]["p@50"]
print(f"\nreported headline stays the honest one: random forest precision@50 = "
      f"{rf_grouped['mean']:.3f} +/- {rf_grouped['std']:.3f}")
print(f"per-fold spread: {rf_grouped['per_fold']}")

base rate            : 0.5421
clients              : 32   rows: 30,000
naive  split         : StratifiedKFold(n_splits=5, shuffle=True)
  test clients seen in training: 100%
grouped split        : GroupKFold(n_splits=5) on client_id
  test clients seen in training: 0%

--- what the naive split would have let me claim ---
               model            metric  honest (grouped)  naive (random)     gap  inflation %
 logistic_regression           roc_auc            0.6605          0.7172  0.0567         8.59
 logistic_regression average_precision            0.6670          0.7333  0.0663         9.94
 logistic_regression              p@50            0.7880          0.8960  0.1080        13.71
       random_forest           roc_auc            0.6707          0.7744  0.1037        15.46
       random_forest average_precision            0.6798          0.7914  0.1116        16.42
       random_forest              p@50            0.7760          0.9520  0.1760        22.68
decision_tree_depth

**Result.** The naive split inflates every learned model on every metric, and it inflates the
most capable model most: random forest precision@50 goes from **0.776 honest to 0.952 naive** — a
22.7% relative overstatement, and the kind of number that would have read as a finished product.

The pattern across the four scorers is the tell, and it is why the ML-07 rule was included as a
control:

| scorer | capacity to memorise a client | precision@50 inflation |
|---|---|---|
| ML-07 hand rule | none — fixed weights, never fitted | **−9.7%** |
| decision tree, depth 2 | two splits | +1.9% |
| logistic regression | linear in 26 features | +13.7% |
| random forest | 300 trees | **+22.7%** |

**The inflation is monotone in model capacity.** The hand rule cannot benefit at all — it is never
fitted, so its small negative movement is pure fold-composition noise and sets the scale for what
"no leak" looks like. Each step up in capacity buys more inflation. That ordering is the signature
of client identity leaking through the split; a genuinely better model would not improve *because*
the split got easier.

Honest number, carried forward: **precision@50 = 0.776 ± 0.088** across five held-out client folds.
The standard deviation is large and stays visible — with 32 clients and one of them holding 23% of
the corpus, fold composition varies a lot, and a single point estimate would hide that.

One reproducibility note: the ML-07 rule scores **0.616 ± 0.136** here, matching ML-08's figure to
three decimals under an independently written harness. The two notebooks agree.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Week 3 excluded `trend_direction`, `trend_pct`, and `is_declining_label` **by name**, on the
reasoning that the label is derived from `trend_direction`. That was correct and it was not enough.
This pass ran three checks instead of one, and the third found something the first two missed.

**Check 1 — every numeric column scored alone against the label.** Leakage found by measurement,
not by remembering to exclude a name. Flag anything with AUC ≥ 0.90.

**Check 2 — the shuffled-label null** from section 2, which bounds how much of any signal is the
harness rather than the data.

**Check 3 — reconstruct the label from first principles.** Not "does any feature correlate with the
label," but "can I *rebuild* the label out of what shipped?" This is the check that found it.

### What check 3 found

The feature vector contains `impressions_last_30d` and `impressions_prev_30d`. `trend_direction` is
defined as the 30-day-over-previous-30-day impression change. So those two columns are not
correlated with the label — **they are the label, in unreduced form.**

The reconstruction is exact: **30,000 of 30,000 rows**, no residual. Two details worth recording:

- The paper documents the cut at ±10% (p. 5). The shipped data cuts at **±20%** — recovered
  empirically, since `down` tops out at −20.021%, `stable` spans exactly [−20, +20], and `up`
  starts at +20.025%. At the documented ±10% the reconstruction agrees on only 93.3% of rows; at
  ±20% it agrees on all of them. I am flagging the mismatch rather than quietly using the number
  that works, because a reader following the documented definition would build a different label.
- The 3,388 rows with `impressions_prev_30d == 0` are exactly the `new` and `flat` pages, and none
  of them is ever labelled declining. That is consistent and it is also a second selection effect
  worth naming: pages with no prior window cannot be declining by construction.

### Why the single-feature scan missed it

`impressions_last_30d` alone scores **AUC 0.486** — worse than a coin flip. `impressions_prev_30d`
alone scores **0.621**. Neither would trip any reasonable threshold. Their **ratio** scores
**AUC 1.000**.

That is the transferable lesson from this card: **a one-feature-at-a-time leakage scan cannot see a
leak that lives in an interaction.** The scan is worth running and it is not sufficient. What caught
this was starting from the label's definition and asking which shipped columns appear in it — an
audit of the data dictionary, not of the correlation table.

**Verdict on my Week-5 model: clean, but by luck as much as by design.** Neither column is in the
ML-08 feature set — the cell below asserts it rather than trusting my memory — because the contract
selected features by relevance and those two never made the list. Had I picked features by "what
correlates with the label," I would have shipped a model with a perfect leak in it and a
precision@50 near 1.0 to advertise.

In [3]:
# Section 3 - the three checks, with the exclusion asserted rather than remembered.
from train_refresh_model import CATEGORICAL, NUMERIC  # noqa: E402

recon = audit["label_reconstruction"]
scan = audit["single_feature_auc"]

print("--- check 1: every numeric column scored alone ---")
flagged = pd.DataFrame(scan["flagged_as_leak"])
print(f"columns flagged at AUC >= 0.90 : {len(flagged)}")
print(flagged.to_string(index=False) if len(flagged) else "  (none)")
print(f"columns in the 0.65-0.90 watch band : {len(scan['flagged_as_watch'])}")
print("\ntop separating columns overall:")
print(pd.DataFrame(scan["all_features"][:8]).to_string(index=False))
print("-> only trend_pct trips it, and trend_pct is the label's own source column.")
print("   Every feature actually in the model sits between 0.41 and 0.59. Scan says clean.")

print("\n--- check 3: rebuild the label from what shipped ---")
print(f"documented rule : {recon['documented_rule']}")
print(f"recovered rule  : {recon['recovered_rule']}")
print("\nboundaries observed on the recomputed 30d change:")
for direction, band in recon["observed_bands_on_recomputed_change"].items():
    print(f"  {direction:7} n={band['n']:6,}  from {band['min_pct']:>10.3f}% to {band['max_pct']:>10.3f}%")
print("\nagreement with the real label, by threshold:")
for threshold, agreement in recon["agreement_by_threshold"].items():
    print(f"  {threshold:>5} -> {agreement:.6f}")
print(f"\nexact reconstruction : {recon['rows_reconstructed_exactly']:,} / {recon['rows_total']:,} rows")
print(f"rows with prev_30d==0: {recon['rows_with_zero_prev_30d']:,} "
      f"(new/flat, never labelled declining: {recon['zero_prev_30d_are_new_or_flat_and_never_labelled_down']})")

print("\n--- why check 1 could not have found it ---")
for column, auc in recon["roc_auc_of_each_column_alone"].items():
    print(f"  {column:<24} alone : AUC {auc:.4f}")
print(f"  {'their ratio':<24}       : AUC {recon['roc_auc_of_impression_ratio_alone']:.4f}")

# The exclusion, asserted. If a future edit adds either column, this notebook fails loudly.
model_features = set(NUMERIC + CATEGORICAL)
leaking = {"impressions_last_30d", "impressions_prev_30d", "trend_direction", "trend_pct",
           "is_declining_label"}
found = sorted(model_features & leaking)
assert not found, f"LEAK: label-reconstructing columns are in the model feature set: {found}"
print(f"\n[OK] none of {sorted(leaking)}")
print(f"     appears in the {len(model_features)} ML-08 model features. Verdict: {recon['verdict']}.")

--- check 1: every numeric column scored alone ---
columns flagged at AUC >= 0.90 : 1
  feature  auc  separation  rows_scored  in_model  is_label_source
trend_pct  0.0         0.5        30000     False             True
columns in the 0.65-0.90 watch band : 0

top separating columns overall:
               feature    auc  separation  rows_scored  in_model  is_label_source
             trend_pct 0.0000      0.5000        30000     False             True
  impressions_prev_30d 0.6214      0.1214        30000     False             True
      content_age_days 0.4085      0.0915        30000      True            False
        age_tier_order 0.4149      0.0851        30000     False            False
       impressions_90d 0.5845      0.0845        30000     False            False
   log_impressions_90d 0.5845      0.0845        30000      True            False
measurable_opportunity 0.5821      0.0821        30000     False            False
 days_with_impressions 0.5794      0.0794        30

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### The sentence, quoted from my own Week-5 notebook

Week 5 has a careful "claims at the strength the evidence supports" section at the end. The bold
sentence is not in it — it is in the prose above it, which is exactly where this kind of thing
survives:

> **"The learned models beat the rule, and they beat it where the rule actually failed. […] That is
> the difference between 'a way to find 50 pages' and 'a queue you can actually work through', and
> it is the single result that justifies moving off the rule."**

### What is wrong with it

| Problem | Where it hides |
|---|---|
| **"beat" (twice)** | A verb of contest. Three of the four scorers here scored higher than the rule on some metric under some split; "beat" compresses a distribution into a verdict. |
| **"a queue you can actually work through"** | Slides from a measured quantity into a workflow outcome. Nobody worked through a queue. Precision@1000 measures how many of the top 1,000 ranked pages carry the declining label — not whether a reviewer's day went better. |
| **"the single result that justifies…"** | Decision language resting on one metric on one corpus under one split, with no statement of what would overturn it. |
| **"where the rule actually failed"** | This one is defensible and the code cell below confirms it — but only because it was checked, not because it was written confidently. |

The second row is the one that matters, and it is not a wording problem. It is the same move I
questioned in Finding B on p. 9 of the paper: a metric about ranking, reported as a claim about an
outcome. Catching myself making it three weeks after criticising it is the point of this section.

### Checking the sentence before rewriting it

The claim decomposes into two testable parts, and **they do not both survive:**

- *at depth* — the forest leads the rule on precision@1000 in **5 of 5 folds**. Holds.
- *at the top of the queue* — the forest leads the rule on precision@50 in **4 of 5 folds**. On
  fold 5 the rule wins, 0.76 to 0.62.

My first draft of the rewrite below said the forest "held an advantage on every fold." The assertion
in the code cell failed on it. That is the whole argument for putting claims under `assert` instead
of under proofreading: the sentence read as careful and was still wrong.

### The rewrite

> **Under a client-grouped 5-fold split on the 30,000-row active-content starter sample, the random
> forest ranked declining pages ahead of the ML-07 hand rule deep into the queue: precision@1000 of
> 0.726 ± 0.054 versus 0.558 ± 0.067, against a base rate of 0.542, with the forest ahead on all
> five held-out client folds. At the top of the queue the advantage is directional rather than
> consistent — precision@50 of 0.776 ± 0.088 versus 0.616 ± 0.136, with the forest ahead on four of
> five folds and the two ranges overlapping. The measured difference is therefore in *sustained*
> ranking depth, not in top-50 accuracy. This is decision-support for ordering a human review
> queue. It is not evidence that refreshing the selected pages recovers traffic — no refresh outcome
> is observed anywhere in this data, and the sample contains only pages that were still active.**

Longer, and every added clause is carrying weight: the split, the sample frame, the ±, the base
rate, the fold counts, which half of the claim is consistent and which is only directional, and the
boundary of what the number licenses.

### The three claims this audit changed

1. **Any headline from a random split is retired.** Precision@50 = 0.952 is a real computation and a
   false claim. The honest figure is 0.776 ± 0.088 and it is the only one that goes in the capstone.
2. **"No leakage" becomes "leakage audited, one exact reconstruction found and excluded."** The
   first is a promise. The second is a finding with a receipt in
   `work/outputs/validation_audit.json`, and it is the stronger sentence precisely because it
   admits the corpus contained a perfect leak.
3. **Every result inherits a survivorship caveat.** The starter CSV is 100% active content. The
   model has never seen a page that already died, so it cannot rank one, and the capstone says so
   next to the metric rather than in a limitations section at the end.

### Words I am no longer using about this work

`beats`, `proves`, `drives`, `causes`, `wide margin`, `significant` (nothing here is significance-tested),
`accurate` unqualified — replaced with: `observed`, `measured`, `ranked`, `directional`,
`decision-support`, `on this sample`, `under this split`.

In [4]:
# Section 4 - every clause of the rewritten claim, asserted against its own numbers.
metrics = comparison["grouped_by_client"]["metrics"]


def fold_wins(metric: str) -> tuple[dict, dict, list[bool]]:
    rf_block = metrics["random_forest"][metric]
    rule_block = metrics["baseline_rule"][metric]
    wins = [a > b for a, b in zip(rf_block["per_fold"], rule_block["per_fold"])]
    return rf_block, rule_block, wins


print("every number in the rewritten claim, traced to its receipt:")
print(f"  split      : {comparison['grouped_by_client']['type']}")
print(f"  sample     : {audit['rows']:,} rows, {audit['n_clients']} clients, 100% active content")
print(f"  base rate  : {audit['base_rate']:.3f}")

for metric, phrase in [("p@1000", "deep into the queue"), ("p@50", "top of the queue")]:
    rf_block, rule_block, wins = fold_wins(metric)
    print(f"\n--- {metric}: {phrase} ---")
    print(f"  random forest : {rf_block['mean']:.3f} +/- {rf_block['std']:.3f}   {rf_block['per_fold']}")
    print(f"  ML-07 rule    : {rule_block['mean']:.3f} +/- {rule_block['std']:.3f}   {rule_block['per_fold']}")
    rf_low, rf_high = rf_block["mean"] - rf_block["std"], rf_block["mean"] + rf_block["std"]
    rule_high = rule_block["mean"] + rule_block["std"]
    print(f"  forest range  : [{rf_low:.3f}, {rf_high:.3f}]   overlaps rule: {rf_low < rule_high}")
    print(f"  forest ahead on {sum(wins)}/{len(wins)} folds")

# The rewrite makes exactly two fold-count claims. Both are asserted, not proofread.
_, _, deep_wins = fold_wins("p@1000")
_, _, top_wins = fold_wins("p@50")
assert all(deep_wins), "rewrite says 'ahead on all five folds' at p@1000 - it is not"
assert sum(top_wins) == 4, f"rewrite says 'four of five folds' at p@50 - actual: {sum(top_wins)}"

# And the boundary clause: no refresh outcome exists in this data to observe.
frame_columns = set(frame.columns)
outcome_like = {c for c in frame_columns if "after" in c.lower() or "post_refresh" in c.lower()}
assert not outcome_like, f"an outcome column exists after all: {outcome_like}"

print("\n[OK] both fold-count claims hold, and no post-refresh outcome column exists to contradict")
print("     the sentence's final clause.")
print(f"     receipt: work/outputs/validation_audit.json  (card {audit['card']}, seed {audit['seed']})")

every number in the rewritten claim, traced to its receipt:
  split      : GroupKFold(n_splits=5) on client_id
  sample     : 30,000 rows, 32 clients, 100% active content
  base rate  : 0.542

--- p@1000: deep into the queue ---
  random forest : 0.726 +/- 0.054   [0.712, 0.784, 0.631, 0.735, 0.768]
  ML-07 rule    : 0.558 +/- 0.067   [0.498, 0.65, 0.466, 0.593, 0.585]
  forest range  : [0.672, 0.780]   overlaps rule: False
  forest ahead on 5/5 folds

--- p@50: top of the queue ---
  random forest : 0.776 +/- 0.088   [0.86, 0.86, 0.78, 0.76, 0.62]
  ML-07 rule    : 0.616 +/- 0.136   [0.5, 0.78, 0.44, 0.6, 0.76]
  forest range  : [0.688, 0.864]   overlaps rule: True
  forest ahead on 4/5 folds

[OK] both fold-count claims hold, and no post-refresh outcome column exists to contradict
     the sentence's final clause.
     receipt: work/outputs/validation_audit.json  (card ML-09, seed 42)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — clients appear only as the anonymised `client_id` used for grouping
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Carried into ML-10 and the capstone:**

1. Headline metrics are **precision@1000 = 0.726 ± 0.054** (forest ahead on 5/5 folds — the
   consistent result) and **precision@50 = 0.776 ± 0.088** (ahead on 4/5), client-grouped, base
   rate 0.542. The naive-split 0.952 is retired.
2. The corpus contains an **exact label reconstruction** (`impressions_last_30d` /
   `impressions_prev_30d`); it is excluded structurally and the exclusion is asserted in code.
3. Trend direction is cut at **±20%** in the data, not the ±10% the paper documents — anything
   built on the documented figure will not reproduce.
4. Every result is on **active content only**; the model cannot rank a page that already died.